# Portuguese Corporate Board Network — 2024

Replicates the methodology of **Battiston & Catanzaro (2004)**  
*"Statistical Properties of Corporate Board and Director Networks"*

Dataset: `boardmembers_2024.csv` — 86 companies, 664 directors, 757 board memberships.

In [ ]:
import os, math, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import matplotlib.pyplot as plt
from scipy.stats import poisson

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 12,
})
os.makedirs('results', exist_ok=True)
LABEL = 'PT 2024'
print('Libraries loaded OK')

## Step 1 — Load data & build the bipartite graph

The raw data is an incidence matrix C (M directors × N companies).  
C[α][i] = 1 if director α sits on board i.  
This is **by definition a bipartite graph**: two disjoint node sets (companies, directors) connected only across sets.

In [ ]:
df = pd.read_csv('boardmembers_2024.csv')
df = df.rename(columns={'name': 'director'})
df = df.dropna(subset=['director', 'company']).drop_duplicates(subset=['director', 'company'])

companies_list = df['company'].unique().tolist()
directors_list = df['director'].unique().tolist()

G_bip = nx.Graph()
G_bip.add_nodes_from(companies_list, bipartite=0)
G_bip.add_nodes_from(directors_list, bipartite=1)
for _, row in df.iterrows():
    G_bip.add_edge(row['company'], row['director'])

companies = {n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 0}
directors = {n for n, d in G_bip.nodes(data=True) if d.get('bipartite') == 1}

print(f'Companies : {len(companies)}')
print(f'Directors : {len(directors)}')
print(f'Edges     : {G_bip.number_of_edges()}')

## Step 2 — One-mode projections

- **Board network B** = C^T · C → companies as nodes; edge weight = # shared directors  
- **Director network D** = C · C^T → directors as nodes; edge weight = # shared boards

In [ ]:
board_net    = bipartite.weighted_projected_graph(G_bip, companies)
director_net = bipartite.weighted_projected_graph(G_bip, directors)

print(f'Board network    : {board_net.number_of_nodes()} nodes, {board_net.number_of_edges()} edges')
print(f'Director network : {director_net.number_of_nodes()} nodes, {director_net.number_of_edges()} edges')

## Step 3 — Table 1: Network metrics

| Column | Definition |
|--------|------------|
| N | total nodes |
| E | total edges |
| Nc/N | fraction of nodes in giant component |
| k/kc (%) | mean degree / (N-1) × 100 |
| b/N | mean betweenness (unnormalised) / N |
| C | mean clustering coefficient |
| d | mean shortest path in giant component |

In [ ]:
def compute_metrics(G, label):
    N  = G.number_of_nodes()
    E  = G.number_of_edges()
    largest_cc = max(nx.connected_components(G), key=len)
    Nc = len(largest_cc)
    G_giant = G.subgraph(largest_cc).copy()
    k_avg = np.mean([d for _, d in G.degree()])
    bet   = nx.betweenness_centrality(G_giant, normalized=False)
    b_avg = np.mean(list(bet.values()))
    C_bar = nx.average_clustering(G)
    if G_giant.number_of_nodes() > 2000:
        sample = random.sample(list(G_giant.nodes()), 500)
        lens   = []
        for s in sample:
            lens.extend(nx.single_source_shortest_path_length(G_giant, s).values())
        d_avg = np.mean(lens)
    else:
        d_avg = nx.average_shortest_path_length(G_giant)
    return {
        'label': label, 'N': N, 'E': E,
        'Nc/N': round(Nc/N, 3),
        'k/kc (%)': round((k_avg/(N-1))*100, 3),
        'b/N': round(b_avg/N, 3),
        'C': round(C_bar, 3),
        'd': round(d_avg, 2),
        '_k_avg': k_avg, '_G_giant': G_giant, '_Nc': Nc, '_bet': bet,
    }

m_board = compute_metrics(board_net,    f'{LABEL} Board')
m_dir   = compute_metrics(director_net, f'{LABEL} Director')

paper = [
    {'label': "Board IT'86",    'N':  221, 'E':  1295, 'Nc/N': 0.97, 'k/kc (%)': 5.29, 'b/N': 0.736, 'C': 0.356, 'd': 3.6},
    {'label': "Board IT'02",    'N':  240, 'E':   636, 'Nc/N': 0.82, 'k/kc (%)': 2.22, 'b/N': 0.875, 'C': 0.318, 'd': 4.4},
    {'label': "Board US'99",    'N':  916, 'E':  3321, 'Nc/N': 0.87, 'k/kc (%)': 1.57, 'b/N': 1.080, 'C': 0.376, 'd': 4.6},
    {'label': "Director IT'86",'N': 2378, 'E': 23603, 'Nc/N': 0.92, 'k/kc (%)': 0.84, 'b/N': 1.116, 'C': 0.899, 'd': 2.7},
    {'label': "Director IT'02",'N': 1906, 'E': 12815, 'Nc/N': 0.84, 'k/kc (%)': 0.71, 'b/N': 1.206, 'C': 0.915, 'd': 3.6},
    {'label': "Director US'99",'N': 7680, 'E': 55437, 'Nc/N': 0.89, 'k/kc (%)': 0.79, 'b/N': 1.384, 'C': 0.884, 'd': 3.7},
]

keys = ['label','N','E','Nc/N','k/kc (%)','b/N','C','d']
rows = [{k: m[k] for k in keys} for m in [m_board, m_dir]] + paper
table1 = pd.DataFrame(rows)
table1.to_csv('results/table1.csv', index=False)
table1.style.set_caption('Table 1 — Network properties')

## Step 4 — Small World check

A network is a **Small World** (Watts-Strogatz) when:  
- Real clustering C >> random clustering C_random ≈ p = 2E/N(N-1)  
- Real path length d ≈ d_random = log(N)/log(k_avg)

In [ ]:
def small_world(m, label):
    N, E  = m['N'], m['E']
    p     = (2*E)/(N*(N-1))
    C_r   = p
    d_r   = math.log(N)/math.log(m['_k_avg']) if m['_k_avg'] > 1 else float('inf')
    ratio = m['C']/C_r if C_r > 0 else float('inf')
    is_sw = ratio > 10 and abs(m['d']-d_r)/d_r < 1.0
    print(f"{label}")
    print(f"  C_real={m['C']:.3f}  C_random={C_r:.4f}  ratio={ratio:.1f}x")
    print(f"  d_real={m['d']:.2f}   d_random={d_r:.2f}")
    print(f"  => Small World: {'YES ✓' if is_sw else 'NO'}")

small_world(m_board, 'Board network')
print()
small_world(m_dir,   'Director network')

## Step 5 — Edge weight distributions (Figs 3 & 4)

- **Director network**: how many boards do pairs of directors co-serve?  
- **Board network**: how many directors do pairs of boards share?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, G, title, fname in [
    (axes[0], director_net, 'Fig 3 — Edge weights: Director network', 'fig3_edge_weights_director.png'),
    (axes[1], board_net,    'Fig 4 — Edge weights: Board network',    'fig4_edge_weights_board.png'),
]:
    weights = [d['weight'] for _, _, d in G.edges(data=True)]
    cnt = Counter(weights)
    xs  = sorted(cnt); ys = [cnt[x] for x in xs]
    ax.semilogy(xs, ys, 'o-', color='steelblue', label=LABEL)
    ax.set_xlabel('w  (shared directors/boards)')
    ax.set_ylabel('# edges  (log)')
    ax.set_title(title)
    ax.legend()
    fig.savefig(f'results/{fname}', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## Step 6 — Chairs distribution: boards per director (Fig 5)

Most directors sit on only 1 board. Directors on 5+ boards deviate from a Poisson expectation — they are **non-random connectors**.

In [ ]:
chairs  = {d: G_bip.degree(d) for d in directors}
total_c = sum(G_bip.degree(c) for c in companies)
ratio_c = total_c / len(directors)
print(f'Chairs ratio = {ratio_c:.4f}  (paper: US=1.23, IT02=1.28, IT86=1.36)')

cnt  = Counter(chairs.values())
xs   = sorted(cnt); ys = [cnt[x] for x in xs]
lam  = ratio_c - 1
k_r  = range(1, max(xs)+2)
pois = [poisson.pmf(k, lam)*len(directors) for k in k_r]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(xs, ys, 'o', color='steelblue', label=LABEL, markersize=7)
ax.loglog(list(k_r), pois, '--', color='salmon', label=f'Poisson(λ={lam:.2f})')
ax.set_xlabel('Number of boards  (log)')
ax.set_ylabel('Number of directors  (log)')
ax.set_title('Fig 5 — Chairs distribution')
ax.legend()
fig.savefig('results/fig5_chairs_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Board size distribution (Fig 6)

How many directors does each board have? Paper: mean ~10, max ~30.

In [ ]:
board_sizes = [G_bip.degree(c) for c in companies]
print(f'Mean board size = {np.mean(board_sizes):.2f}  |  Max = {max(board_sizes)}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(board_sizes, bins=range(1, max(board_sizes)+2),
        density=True, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Board size (number of directors)')
ax.set_ylabel('P(n) — fraction of boards')
ax.set_title('Fig 6 — Board size distribution')
fig.savefig('results/fig6_board_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — Cumulative degree distributions CCDF (Figs 7 & 8)

P'(k) = fraction of nodes with degree ≥ k.  
Director network: expect a **plateau up to k≈10** (typical board size).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, G, title, fname in [
    (axes[0], director_net, "Fig 7 — Degree CCDF: Director network", 'fig7_degree_dist_director.png'),
    (axes[1], board_net,    "Fig 8 — Degree CCDF: Board network",    'fig8_degree_dist_board.png'),
]:
    degs  = sorted([d for _, d in G.degree()], reverse=True)
    cumul = [i/len(degs) for i in range(1, len(degs)+1)]
    ax.semilogy(degs, cumul, 'o', color='steelblue', markersize=4, label=LABEL)
    ax.set_xlabel('k (degree)')
    ax.set_ylabel("P'(k)  (log)")
    ax.set_title(title)
    ax.legend()
    fig.savefig(f'results/{fname}', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## Step 9 — Betweenness distributions (Figs 9 & 10)

Cumulative distribution of normalised betweenness b/N. Paper finds near-exponential decay.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, m, title, fname in [
    (axes[0], m_dir,   'Fig 9 — Betweenness CCDF: Director', 'fig9_betweenness_dist_director.png'),
    (axes[1], m_board, 'Fig 10 — Betweenness CCDF: Board',   'fig10_betweenness_dist_board.png'),
]:
    bet_vals = sorted([v/m['N'] for v in m['_bet'].values()], reverse=True)
    cumul    = [i/len(bet_vals) for i in range(1, len(bet_vals)+1)]
    ax.semilogy(bet_vals, cumul, 'o', color='steelblue', markersize=4, label=LABEL)
    ax.set_xlabel('b/N')
    ax.set_ylabel("P'(b/N)  (log)")
    ax.set_title(title)
    ax.legend()
    fig.savefig(f'results/{fname}', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## Step 10 — Betweenness vs degree (Figs 11 & 12)

Power-law correlation on log-log scale.  
Paper finds slope ≈ 2.2 (directors) and ≈ 1.5 (boards).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, m, title, fname, exp_slope in [
    (axes[0], m_dir,   'Fig 11 — b/N vs k: Director', 'fig11_bet_vs_degree_director.png', 2.2),
    (axes[1], m_board, 'Fig 12 — b/N vs k: Board',    'fig12_bet_vs_degree_board.png',    1.5),
]:
    G_g = m['_G_giant']
    xs  = np.array([G_g.degree(n) for n in G_g.nodes()])
    ys  = np.array([m['_bet'][n]/m['N'] for n in G_g.nodes()])
    mask = (xs > 0) & (ys > 0)
    c   = np.polyfit(np.log(xs[mask]), np.log(ys[mask]), 1)
    xl  = np.linspace(xs[mask].min(), xs[mask].max(), 100)
    yl  = np.exp(c[1]) * xl**c[0]
    ax.loglog(xs, ys, 'o', color='steelblue', markersize=4, alpha=0.7, label=LABEL)
    ax.loglog(xl, yl, '--', color='salmon', label=f'fit slope={c[0]:.2f} (paper~{exp_slope})')
    ax.set_xlabel('k (degree, log)')
    ax.set_ylabel('b/N (log)')
    ax.set_title(title)
    ax.legend()
    fig.savefig(f'results/{fname}', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## Step 11 — Table 2: Assortativity

Newman's assortativity coefficient r.  
r > 0 = **assortative**: high-degree nodes connect to high-degree nodes (typical of social networks).

In [ ]:
r_board = nx.degree_assortativity_coefficient(board_net)
r_dir   = nx.degree_assortativity_coefficient(director_net)

assort = pd.DataFrame([
    {'Network': f'{LABEL} Board',    'r': round(r_board, 3), 'Assortative': r_board > 0},
    {'Network': f'{LABEL} Director', 'r': round(r_dir,   3), 'Assortative': r_dir   > 0},
    {'Network': "Board IT'86",       'r': 0.12, 'Assortative': True},
    {'Network': "Board IT'02",       'r': 0.32, 'Assortative': True},
    {'Network': "Board US'99",       'r': 0.27, 'Assortative': True},
    {'Network': "Director IT'86",    'r': 0.13, 'Assortative': True},
    {'Network': "Director IT'02",    'r': 0.25, 'Assortative': True},
    {'Network': "Director US'99",    'r': 0.27, 'Assortative': True},
])
assort.to_csv('results/table2_assortativity.csv', index=False)
assort.style.set_caption('Table 2 — Assortativity')

## Step 12 — Knn(k): Average Nearest-Neighbour Degree (Figs 13 & 14)

If Knn(k) increases with k → **assortative** network (confirms Table 2).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, G, title, fname in [
    (axes[0], director_net, 'Fig 13 — Knn(k): Director', 'fig13_knn_director.png'),
    (axes[1], board_net,    'Fig 14 — Knn(k): Board',    'fig14_knn_board.png'),
]:
    knn = nx.average_neighbor_degree(G)
    by_k = defaultdict(list)
    for n in G.nodes():
        by_k[G.degree(n)].append(knn[n])
    ks   = sorted(by_k)
    knns = [np.mean(by_k[k]) for k in ks]
    ax.loglog(ks, knns, 'o', color='steelblue', markersize=5, label=LABEL)
    ax.set_xlabel('k (degree, log)')
    ax.set_ylabel('Knn(k) (log)')
    ax.set_title(title)
    ax.legend()
    fig.savefig(f'results/{fname}', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## Step 13 — c(k): Clustering vs degree (Figs 15 & 16)

- If c(k) ~ 1/k → hierarchical social network (Barabási)  
- If c(k) is flat or slow-decaying → **clique structure = lobbies**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, G, title, fname in [
    (axes[0], board_net,    'Fig 15 — c(k): Board',    'fig15_ck_board.png'),
    (axes[1], director_net, 'Fig 16 — c(k): Director', 'fig16_ck_director.png'),
]:
    clust = nx.clustering(G)
    by_k  = defaultdict(list)
    for n in G.nodes():
        k = G.degree(n)
        if k > 0:
            by_k[k].append(clust[n])
    ks  = sorted(by_k)
    cks = [np.mean(by_k[k]) for k in ks]
    k_r = np.array(ks, dtype=float)
    scale = np.mean(cks) / np.mean(1/k_r)
    ax.loglog(ks, cks, 'o', color='steelblue', markersize=5, label=f'c(k) {LABEL}')
    ax.loglog(k_r, scale/k_r, '--', color='salmon', alpha=0.7, label='1/k ref')
    ax.set_xlabel('k (degree, log)')
    ax.set_ylabel('c(k) (log)')
    ax.set_title(title)
    ax.legend()
    fig.savefig(f'results/{fname}', dpi=150, bbox_inches='tight')

plt.tight_layout()
plt.show()

## Step 14 — Lobbies

A **lobby** in board i = 2+ directors of board i who also co-serve in another board j.  
Paper: US=35%, IT'86=44%, IT'02=63%.

In [ ]:
boards_with_lobby = 0
lobby_rows = []

for c in companies:
    dirs = list(G_bip.neighbors(c))
    if len(dirs) < 2:
        continue
    found = False
    for i in range(len(dirs)):
        if found: break
        for j in range(i+1, len(dirs)):
            d1, d2 = dirs[i], dirs[j]
            common = (set(G_bip.neighbors(d1)) - {c}) & (set(G_bip.neighbors(d2)) - {c})
            if common:
                boards_with_lobby += 1
                lobby_rows.append({'company': c, 'director1': d1, 'director2': d2,
                                   'shared_other_boards': ', '.join(common)})
                found = True; break

pct = (boards_with_lobby / len(companies)) * 100
print(f'{boards_with_lobby}/{len(companies)} companies have a lobby >= 2  ({pct:.1f}%)')
print('Paper: US=35%, IT86=44%, IT02=63%')

lobby_df = pd.DataFrame(lobby_rows)
lobby_df.to_csv('results/lobbies.csv', index=False)
lobby_df.head(10)

## Final Summary

In [ ]:
print('='*62)
print('SUMMARY — comparison with Battiston & Catanzaro (2004)')
print('='*62)
print('\nTable 1 — Network properties:')
print(table1.to_string(index=False))
print('\nTable 2 — Assortativity:')
print(assort.to_string(index=False))
print(f'\nLobbies >= 2 : {pct:.1f}%  (paper: US=35%, IT86=44%, IT02=63%)')
print('\nAll figures saved to ./results/')
print('='*62)